### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


c:\Users\utkar\Desktop\projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: DocuMind.pdf
  ✓ Loaded 3 pages

Processing: quiz psy.pdf
  ✓ Loaded 5 pages

Total documents loaded: 8


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'DocuMind', 'source': '..\\data\\pdf\\DocuMind.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'DocuMind.pdf', 'file_type': 'pdf'}, page_content='A  Retrieval-Augmented  Generation  (RAG)  AI  Assistant  \n1.  Project  Overview  \nDocuMind  is  an  intelligent,  local  "Second  Brain"  chatbot  designed  to  help  students  seamlessly  \nnavigate\n \nmassive\n \namounts\n \nof\n \ntext.\n \nUnlike\n \nstandard\n \nAI\n \nmodels\n \nthat\n \ngive\n \ngeneric\n \nanswers\n \nfrom\n \nthe\n \ninternet,\n \nthis\n \nproject\n \ncreates\n \na\n \nclosed-loop\n \nAI\n \nassistant\n \nthat\n \nreads\n \nyour\n \nspecific\n \nfiles\n \n(like\n \ncollege\n \ntextbooks,\n \nlecture\n \nPDFs,\n \nand\n \nnotes)\n \nand\n \nanswers\n \nquestions\n \nbased\n \nstrictly\n \non\n \nthat\n \nmaterial.\n \nBy  the  end  of  this  8-week  summer  project,  mentees  

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 8 documents into 22 chunks

Example chunk:
Content: A  Retrieval-Augmented  Generation  (RAG)  AI  Assistant  
1.  Project  Overview  
DocuMind  is  an  intelligent,  local  "Second  Brain"  chatbot  designed  to  help  students  seamlessly  
navigate
...
Metadata: {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'DocuMind', 'source': '..\\data\\pdf\\DocuMind.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'DocuMind.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'DocuMind', 'source': '..\\data\\pdf\\DocuMind.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'DocuMind.pdf', 'file_type': 'pdf'}, page_content='A  Retrieval-Augmented  Generation  (RAG)  AI  Assistant  \n1.  Project  Overview  \nDocuMind  is  an  intelligent,  local  "Second  Brain"  chatbot  designed  to  help  students  seamlessly  \nnavigate\n \nmassive\n \namounts\n \nof\n \ntext.\n \nUnlike\n \nstandard\n \nAI\n \nmodels\n \nthat\n \ngive\n \ngeneric\n \nanswers\n \nfrom\n \nthe\n \ninternet,\n \nthis\n \nproject\n \ncreates\n \na\n \nclosed-loop\n \nAI\n \nassistant\n \nthat\n \nreads\n \nyour\n \nspecific\n \nfiles\n \n(like\n \ncollege\n \ntextbooks,\n \nlecture\n \nPDFs,\n \nand\n \nnotes)\n \nand\n \nanswers\n \nquestions\n \nbased\n \nstrictly\n \non\n \nthat\n \nmaterial.\n \nBy  the  end  of  this  8-week  summer  project,  mentees  

# embedding and vectoratoreDB
### we will use open source embeddors 

from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2970.72it/s]


Model loaded successfully. Embedding dimension: 384


In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 22


In [11]:
chunks

[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'DocuMind', 'source': '..\\data\\pdf\\DocuMind.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'DocuMind.pdf', 'file_type': 'pdf'}, page_content='A  Retrieval-Augmented  Generation  (RAG)  AI  Assistant  \n1.  Project  Overview  \nDocuMind  is  an  intelligent,  local  "Second  Brain"  chatbot  designed  to  help  students  seamlessly  \nnavigate\n \nmassive\n \namounts\n \nof\n \ntext.\n \nUnlike\n \nstandard\n \nAI\n \nmodels\n \nthat\n \ngive\n \ngeneric\n \nanswers\n \nfrom\n \nthe\n \ninternet,\n \nthis\n \nproject\n \ncreates\n \na\n \nclosed-loop\n \nAI\n \nassistant\n \nthat\n \nreads\n \nyour\n \nspecific\n \nfiles\n \n(like\n \ncollege\n \ntextbooks,\n \nlecture\n \nPDFs,\n \nand\n \nnotes)\n \nand\n \nanswers\n \nquestions\n \nbased\n \nstrictly\n \non\n \nthat\n \nmaterial.\n \nBy  the  end  of  this  8-week  summer  project,  mentees  

## convert text to embeddings


In [12]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['A  Retrieval-Augmented  Generation  (RAG)  AI  Assistant  \n1.  Project  Overview  \nDocuMind  is  an  intelligent,  local  "Second  Brain"  chatbot  designed  to  help  students  seamlessly  \nnavigate\n \nmassive\n \namounts\n \nof\n \ntext.\n \nUnlike\n \nstandard\n \nAI\n \nmodels\n \nthat\n \ngive\n \ngeneric\n \nanswers\n \nfrom\n \nthe\n \ninternet,\n \nthis\n \nproject\n \ncreates\n \na\n \nclosed-loop\n \nAI\n \nassistant\n \nthat\n \nreads\n \nyour\n \nspecific\n \nfiles\n \n(like\n \ncollege\n \ntextbooks,\n \nlecture\n \nPDFs,\n \nand\n \nnotes)\n \nand\n \nanswers\n \nquestions\n \nbased\n \nstrictly\n \non\n \nthat\n \nmaterial.\n \nBy  the  end  of  this  8-week  summer  project,  mentees  will  have  built  a  fully  functional  web  \napplication\n \nthat\n \noperates\n \nlike\n \na\n \ncustom\n \nChatGPT\n \nrestricted\n \nto\n \ntheir\n \nown\n \ncourse\n \nmaterials.\n \n The  8-Week  Implementation  Plan  \nWeek  1:  Laying  the  Foundations  \nSetting  up  the  

In [13]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 22 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.43s/it]

Generated embeddings with shape: (22, 384)
Adding 22 documents to vector store...
Successfully added 22 documents to vector store
Total documents in collection: 44


## Retriever Pipeline From VectorStore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve(" three reasons for forgetting")

Retrieving documents for query: ' three reasons for forgetting'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_6ce68019_8',
  'content': '(serial positioning curve and free recall) \n6. Forgetting that happens due to response competition is called what? (interference) \n7. Describe any three reasons for forgetting?  \n(encoding failure, decay of memory trace, lack of consolidation, interference, repression, \nor motivated forgetting) \n8. In the LOP theory of memory, at which level of processing, the phonetic and visual \ncharacteristics of the input are processed? (Structural) \n9. Reconstructions that occur in people with memory disorders are called ______. \n(Confabulations) \n10. Defensive amnesia is an extreme form of _______.  \n \nINTELLIGENCE \n1. ____ is a measure of innate ability in a particular domain, whereas ______ is \nlearning/training in a particular domain.  \n2. In Sternberg’s Triarchic theory of Intelligence, different components are important for \ndifferent competencies. _______ account for individual differences in fluid intelligence,',
  'metadata': {'produc

### Integration Vectordb Context pipeline With LLM output

In [17]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [18]:
answer=rag_simple("three reasons for forgetting",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'three reasons for forgetting'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Three common reasons for forgetting are:

1. **Encoding failure** – the information was never properly encoded into memory.  
2. **Interference** – competing memories hinder retrieval of the target memory.  
3. **Decay of the memory trace** – the stored memory fades over time if not rehearsed.


### Enhanced RAG Pipeline Features

In [19]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("three reasons for forgetting", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'three reasons for forgetting'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: Three common reasons for forgetting are:

1. **Encoding failure** – the information was never properly encoded into memory.  
2. **Interference** – other memories compete with the target memory, making retrieval difficult.  
3. **Decay of the memory trace** – the stored memory fades over time if it isn’t rehearsed or consolidated.
Sources: [{'source': 'quiz psy.pdf', 'page': 0, 'score': 0.10361576080322266, 'preview': '(serial positioning curve and free recall) \n6. Forgetting that happens due to response competition is called what? (interference) \n7. Describe any three reasons for forgetting?  \n(encoding failure, decay of memory trace, lack of consolidation, interference, repression, \nor motivated forgetting) \n8. ...'}, {'source': 'quiz psy.pdf', 'page': 0, 'score': 0.10361576080322266, 'preview': '(serial positioning curve and free recall) \n6. Forgetting that happens due to response competition is called what? (interference) \n7. Describe any three reasons for forgetting

In [20]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("three reasons for forgetting", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'three reasons for forgetting'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
(serial positioning curve and free recall) 
6. Forgetting that happens due to response competition is called what? (interference) 
7. Describe any three reasons for forgetting?  
(encoding failure, decay of memory trace, lack of consolidation, interfe

rence, repression, 
or motivated forgetting) 
8. In the LOP theory of memory, at which level of processing, the phonetic and visual 
characteristics of the input are processed? (Structural) 
9. Reconstructions that occur in people with memory disorders are called ______. 
(Confabulations) 
10. Defensive amnesia is an extreme form of _______.  
 
INTELLIGENCE 
1. ____ is a measure of innate ability in a particular domain, whereas ______ is 
learning/training in a particular domain.  
2. In Sternberg’s Triarchic theory of Intelligence, different components are important for 
different competencies. _______ account for individual differences in fluid intelligence,

(serial positioning curve and free recall) 
6. Forgetting that happens due to response competition is called what? (interference) 
7. Describe any three reasons for forgetting?  
(encoding failure, decay of memory trace, lack of consolidation, interference, repression, 
or motivated forgetting) 
8. In the LOP theory of memory, 